# multi_1h_strat_036

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (25).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 12 |
| Detected functions | resample_1hr_custom, cfba_jdmx_histogram, process_ticker, backtest_trades, calculate_metrics, apply_strategy, zero_lag_ma, detrend_with_zlma, goertzel, detect_strongest_cycle, generate_signal_wave |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/5min_data.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2024-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.signal import find_peaks
import numpy as np
import pandas as pd

def cfba_jdmx_histogram(df,
                        length=32,
                        phs=0,
                        siglen=5,
                        nlen=50,
                        cfb_len=8,
                        smth=8,
                        slim=10,
                        llim=60,
                        jcfbsmlen=10,
                        jcfbsmph=0,
                        kama_fast=0.666,
                        kama_slow=0.0645):
    """
    Compute CFB-Adaptive Jurik DMX Histogram with KAMA smoothing in one function.

    df: pd.DataFrame with 'close' column.
    Returns df with 'histogram', 'signal_line', 'signal' columns.
    """
    price = df['close']

    # --- Step 1: KAMA calculation ---
    change = price.diff(length).abs()
    volatility = price.diff().abs().rolling(length).sum()
    er = change / volatility
    er.replace([np.inf, -np.inf], 0, inplace=True)
    er.fillna(0, inplace=True)

    fast_sc = 2 / (kama_fast * 100 + 1)
    slow_sc = 2 / (kama_slow * 100 + 1)
    sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2

    kama = np.zeros(len(price))
    kama[0] = price.iloc[0]

    for i in range(1, len(price)):
        kama[i] = kama[i - 1] + sc.iloc[i] * (price.iloc[i] - kama[i - 1])
    kama = pd.Series(kama, index=price.index)

    # --- Step 2: CFB Adaptive Filter ---
    filt = np.zeros(len(price))
    filt[0] = kama.iloc[0]

    for i in range(1, len(price)):
        diff = kama.iloc[i] - filt[i - 1]

        # Apply limits (llim)
        if diff > llim:
            diff = llim
        elif diff < -llim:
            diff = -llim

        filt[i] = filt[i - 1] + diff / nlen  # normalize by nlen

        # Simple smoothing averaging last cfb_len values
        if i >= cfb_len:
            filt[i] = np.mean(filt[i - cfb_len + 1:i + 1])
    filt = pd.Series(filt, index=price.index)

    # --- Step 3: Signal smoothing ---
    signal_line = filt.rolling(window=siglen, min_periods=1).mean()

    # --- Step 4: Histogram ---
    histogram = filt - signal_line

    # --- Step 5: Signal generation ---
    signal = np.where(histogram > 0, 1, np.where(histogram < 0, 2, 0))

    # Append results to DataFrame
    df = df.copy()
    df['histogram'] = histogram
    df['signal_line'] = signal_line
    df['Signal'] = signal

    return df

# === TRADE PROCESSOR ===
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time, start_trading_time):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp = row['Date']
        close = row['close']
        open_price = row['open']
        high = row['high']
        low = row['low']
        signal = row['Signal']
        time_str = pd.to_datetime(timestamp).strftime('%H:%M')

        # === Exit Logic ===
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price = position["entry_price"]
            direction = position["direction"]
            qty = position["qty"]

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            exit_price, exit_reason = None, None

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif time_str >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                trades.append([
                    ticker, position["entry_time"], timestamp, entry_price,
                    exit_price, pnl, exit_reason, cumulative_pnl, direction
                ])

                del open_positions[ticker]
                continue

        # === Entry Logic ===
        if ticker not in open_positions and start_trading_time < time_str < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price

            open_positions[ticker] = {
                "entry_time": timestamp,
                "entry_price": entry_price,
                "direction": direction,
                "qty": qty
            }

    return trades

# === BACKTEST EXECUTION ===
def backtest_trades(data, tp_pct, sl_pct):
    initial_capital = 100_000
    brokerage = 0.001
    slippage = 0.0001
    end_time = "15:10"
    stop_trading_time = "14:20"
    start_trading_time = "09:10"

    unique_tickers = data['Ticker'].unique()

    data_dict = {
        ticker: df.to_dict('records')
        for ticker, df in data.groupby('Ticker')
    }

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker, data_dict[ticker], initial_capital, brokerage, slippage,
            tp_pct, sl_pct, end_time, stop_trading_time, start_trading_time
        )
        for ticker in tqdm(unique_tickers)
    )

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

# === MAIN EXECUTION ===

# Convert Date to datetime
data['Date'] = pd.to_datetime(data['Date'])

# Apply signal generation
data = data.groupby('Ticker', group_keys=False).apply(cfba_jdmx_histogram)

# Shift signal for execution on next bar
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(int)

# Backtest
tradebook = backtest_trades(data, tp_pct=0.05, sl_pct=0.01)

# Save results
tradebook.to_csv("/content/drive/MyDrive/tradebook_CYCLE.csv", index=False)

# Output preview
tradebook


In [ ]:
from itertools import product
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# === Metrics Calculation Function (with R² and Monthly Consistency) ===
def calculate_metrics(tradebook):
    if tradebook.empty:
        return 0, 0, 0, 0

    # Ensure datetime
    tradebook['Date'] = pd.to_datetime(tradebook['Entry Time'])  # Replace with correct column if different

    equity = tradebook['PnL'].cumsum()
    total_profit = equity.iloc[-1]

    roll_max = equity.cummax()
    drawdown = roll_max - equity
    max_drawdown = drawdown.max()

    if max_drawdown == 0:
        ratio = float('inf') if total_profit > 0 else 0
    else:
        ratio = total_profit / max_drawdown

    # R² for equity curve smoothness
    x = np.arange(len(equity)).reshape(-1, 1)
    y = equity.values.reshape(-1, 1)
    model = LinearRegression().fit(x, y)
    r2 = model.score(x, y)

    # Monthly return consistency metric
    tradebook['Month'] = tradebook['Date'].dt.to_period('M')
    monthly_returns = tradebook.groupby('Month')['PnL'].sum()
    if len(monthly_returns) < 2:
        monthly_consistency = 0
    else:
        monthly_mean = monthly_returns.mean()
        monthly_std = monthly_returns.std()
        monthly_consistency = monthly_mean / monthly_std if monthly_std > 0 else float('inf')

    return ratio, total_profit, r2, monthly_consistency

# === Hyperparameter Grid ===
param_grid = {
    'length': [32],
    'siglen': [3, 7, 15],
    'nlen': [30, 50, 80],
    'smth': [8],
    'slim': [ 5, 10, 15],
    'llim': [50, 60 , 70],
    'cfb_len': [8],
    'kama_fast': [0.3, 0.666, 0.8, 1],
    'kama_slow': [0.01, 0.03, 0.0645, 0.08, 0.1],
    'tp_pct': [0.02, 0.03, 0.05],
    'sl_pct': [0.001, 0.01]
}

# === Generate All Parameter Combinations ===
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
results = []

print(f"Total combinations to test: {len(param_combinations)}\n")

# === Hyperparameter Tuning Loop ===
for i, combo in enumerate(tqdm(param_combinations, desc="Hypertuning"), 1):
    param = dict(zip(keys, combo))

    print(f"\n=== [{i}/{len(param_combinations)}] Testing Parameters ===")
    for k, v in param.items():
        print(f"{k}: {v}")

    temp_data = data.copy()

    def apply_strategy(group):
        return cfba_jdmx_histogram(
            group,
            length=param['length'],
            siglen=param['siglen'],
            nlen=param['nlen'],
            cfb_len=param['cfb_len'],
            smth=param['smth'],
            slim=param['slim'],
            llim=param['llim'],
            kama_fast=param['kama_fast'],
            kama_slow=param['kama_slow']
        )

    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

    # Shift signals to next bar to avoid lookahead bias
    temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
    temp_data = temp_data.dropna(subset=['Signal'])
    temp_data['Signal'] = temp_data['Signal'].astype(int)

    # Run Backtest
    tradebook = backtest_trades(
        temp_data,
        tp_pct=param['tp_pct'],
        sl_pct=param['sl_pct'],
    )

    # Evaluate metrics
    ratio, total_profit, r2, monthly_consistency = calculate_metrics(tradebook)

    print(f"➡️ Profit-to-Drawdown Ratio: {ratio:.4f}")
    print(f"➡️ Total Profit: ₹{total_profit:,.2f}")
    print(f"➡️ Equity Curve R²: {r2:.4f}")
    print(f"➡️ Monthly Consistency Score: {monthly_consistency:.4f}")

    result = {
        **param,
        'Profit/Drawdown': ratio,
        'Total Profit': total_profit,
        'Equity R2': r2,
        'Monthly Consistency': monthly_consistency
    }
    results.append(result)

# === Final Results and Export ===
results_df = pd.DataFrame(results)

# Optional: Composite metric for sorting
results_df['Composite Score'] = (
    results_df['Profit/Drawdown'] * 0.6 +
    results_df['Equity R2'] * 0.2 +
    results_df['Monthly Consistency'] * 0.2
)

results_df = results_df.sort_values(by='Composite Score', ascending=False)
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results_adaptive_intraday_full.csv", index=False)

print("\n✅ Hyperparameter tuning complete. Top 5 results:")
print(results_df.head())


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
'''
# --- Zero-Lag Moving Average ---
def zero_lag_ma(series, period):
    weights = np.arange(period, 0, -1)
    lwma1 = series.rolling(period).apply(lambda x: np.dot(weights, x) / weights.sum(), raw=True)
    lwma2 = lwma1.rolling(period).apply(lambda x: np.dot(weights, x) / weights.sum(), raw=True)
    return lwma2

# --- ZLMA Detrending ---
def detrend_with_zlma(series, fast_period, slow_period):
    fast = zero_lag_ma(series, fast_period)
    slow = zero_lag_ma(series, slow_period)
    detrended = fast - slow
    return detrended, fast, slow

# --- Goertzel Algorithm ---
def goertzel(samples, sample_rate, target_freq):
    N = len(samples)
    k = int(0.5 + (N * target_freq) / sample_rate)
    omega = (2 * np.pi * k) / N
    coeff = 2 * np.cos(omega)
    s_prev, s_prev2 = 0.0, 0.0
    for sample in samples:
        s = sample + coeff * s_prev - s_prev2
        s_prev2, s_prev = s_prev, s
    real = s_prev - s_prev2 * np.cos(omega)
    imag = s_prev2 * np.sin(omega)
    magnitude = np.sqrt(real**2 + imag**2)
    phase = np.arctan2(imag, real)
    return magnitude, phase

# --- Find Strongest Cycle ---
def detect_strongest_cycle(series, max_cycle=120, window=100):
    best_mag, best_freq, best_phase = 0, 0, 0
    segment = series[-window:].dropna().values
    for cycle_len in range(5, max_cycle + 1):
        freq = 1 / cycle_len
        mag, phase = goertzel(segment, 1, freq)
        if mag > best_mag:
            best_mag, best_freq, best_phase = mag, freq, phase
    return best_mag, best_freq, best_phase

# --- Signal Generation ---
def generate_signal_wave(freq, phase, length):
    t = np.arange(length)
    return np.sin(2 * np.pi * freq * t + phase)

# --- Main Pipeline ---
fast_period = 5
slow_period = 55
window_size = 120
max_cycle = 175

detrended, fast_zlma, slow_zlma = detrend_with_zlma(data['close'], fast_period, slow_period)
magnitude, frequency, phase = detect_strongest_cycle(detrended, max_cycle, window_size)
signal = generate_signal_wave(frequency, phase, len(data)) if frequency > 0 else np.zeros(len(data))

# Normalize signal to [0, 1]
normalized_signal = (signal - np.min(signal)) / (np.max(signal) - np.min(signal))
data['Signal'] = normalized_signal

# --- Plotting ---
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.02,
                    subplot_titles=('Close Price', 'Detrended (ZLMA Fast - Slow)', 'Signal (Goertzel Cycle)'))

fig.add_trace(go.Scatter(x=data['Date'], y=data['close'], name='Close'), row=1, col=1)
fig.add_trace(go.Scatter(x=data['Date'], y=detrended, name='Detrended'), row=2, col=1)
fig.add_trace(go.Scatter(x=data['Date'], y=data['Signal'], name='Signal'), row=3, col=1)

fig.update_layout(height=800, title_text='Goertzel Cycle Detector with ZLMA', showlegend=False)
fig.show()
'''